# Customer Churn — Modeling
**Kaggle Playground Series S6E3**

### What this notebook does
1. Builds a reusable preprocessing pipeline
2. Trains three models: baseline LR → cost-sensitive LR → XGBoost
3. Evaluates each on ROC-AUC (competition metric) and business cost
4. Generates a Kaggle submission from the best model
5. Saves the best model as a `.pkl` file

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

from src.preprocessing import load_data, encode_target, build_preprocessor, ALL_FEATURE_COLS
from src.model_utils import evaluate, plot_roc_curves, plot_confusion_matrix, find_optimal_threshold, make_submission

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42

TRAIN_PATH  = '../data/raw/train.csv'
TEST_PATH   = '../data/raw/test.csv'
MODEL_DIR   = '../src/models/'
SUB_PATH    = '../data/processed/submission.csv'

## 1. Load & Prepare Data

In [ ]:
train, test = load_data(TRAIN_PATH, TEST_PATH)

X = train[ALL_FEATURE_COLS]
y = encode_target(train['Churn'])
X_test = test[ALL_FEATURE_COLS]
test_ids = test['id']

print(f'X train : {X.shape}')
print(f'X test  : {X_test.shape}')
print(f'Churn rate: {y.mean():.3f}')

## 2. Cross-Validation Setup

Using **5-fold Stratified K-Fold** to preserve class ratios across folds. Each model is evaluated by CV ROC-AUC before we commit to full training.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def cv_auc(pipeline):
    scores = cross_val_score(pipeline, X, y, cv=cv,
                             scoring='roc_auc', n_jobs=-1)
    print(f'  CV AUC: {scores.mean():.4f} ± {scores.std():.4f}')
    return scores

## 3. Model 1 — Baseline Logistic Regression

Logistic regression is interpretable and gives us a meaningful baseline with pseudo R² and coefficient-level insights.

In [ ]:
preprocessor = build_preprocessor()

lr_baseline = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, solver='lbfgs'))
])

print('Logistic Regression (baseline):')
lr_baseline_scores = cv_auc(lr_baseline)

In [ ]:
# Fit on full training set for inspection
lr_baseline.fit(X, y)

# Pseudo R² (McFadden's) and AIC/BIC via statsmodels
import statsmodels.api as sm

X_transformed = lr_baseline.named_steps['prep'].transform(X)
X_sm = sm.add_constant(X_transformed)
logit_model = sm.Logit(y, X_sm)
result = logit_model.fit(maxiter=300, disp=False)

print(f"Pseudo R² (McFadden's): {result.prsquared:.4f}")
print(f"AIC                   : {result.aic:.2f}")
print(f"BIC                   : {result.bic:.2f}")

## 4. Model 2 — Cost-Sensitive Logistic Regression

`class_weight='balanced'` penalises misclassifying the minority class (churners) more heavily, which aligns the model with the business objective: missing a churner is more expensive than a false alarm.

In [ ]:
preprocessor2 = build_preprocessor()

lr_balanced = Pipeline([
    ('prep', preprocessor2),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        solver='lbfgs'
    ))
])

print('Logistic Regression (cost-sensitive):')
lr_balanced_scores = cv_auc(lr_balanced)
lr_balanced.fit(X, y)

## 5. Model 3 — XGBoost

Gradient boosting handles the feature interactions (e.g. high monthly charges + month-to-month contract) that logistic regression misses. We use `scale_pos_weight` to compensate for class imbalance.

In [ ]:
neg_pos_ratio = (y == 0).sum() / (y == 1).sum()
print(f'scale_pos_weight = {neg_pos_ratio:.2f}')

preprocessor3 = build_preprocessor()

xgb = Pipeline([
    ('prep', preprocessor3),
    ('clf', XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=neg_pos_ratio,
        eval_metric='auc',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    ))
])

print('XGBoost:')
xgb_scores = cv_auc(xgb)
xgb.fit(X, y)

## 6. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['LR Baseline', 'LR Cost-Sensitive', 'XGBoost'],
    'CV AUC Mean': [
        lr_baseline_scores.mean(),
        lr_balanced_scores.mean(),
        xgb_scores.mean()
    ],
    'CV AUC Std': [
        lr_baseline_scores.std(),
        lr_balanced_scores.std(),
        xgb_scores.std()
    ]
})

results = results.sort_values('CV AUC Mean', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

## 7. ROC Curves & Confusion Matrix (Validation Fold)

We'll use one held-out fold to generate probability estimates for visual comparison.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Fit on train split, predict on val split
models_val = {}
for name, pipe in [
    ('LR Baseline',       lr_baseline),
    ('LR Cost-Sensitive', lr_balanced),
    ('XGBoost',           xgb)
]:
    pipe.fit(X_tr, y_tr)
    models_val[name] = pipe.predict_proba(X_val)[:, 1]

fig = plot_roc_curves(models_val, y_val)
fig.savefig('../data/processed/fig_roc_curves.png', dpi=150)
plt.show()

In [ ]:
# Full evaluation with cost analysis for each model
for name, proba in models_val.items():
    evaluate(y_val, proba, threshold=0.5, model_name=name)

## 8. Optimal Threshold (Business Cost)

The default 0.5 threshold is rarely optimal. We sweep thresholds to find the one that minimises the total cost of missed churners + wasted retention offers.

In [ ]:
best_model_name = results.iloc[0]['Model']
best_proba_val = models_val[best_model_name]

optimal_t = find_optimal_threshold(y_val, best_proba_val)
print(f'Optimal threshold for {best_model_name}: {optimal_t:.2f}')

evaluate(y_val, best_proba_val, threshold=optimal_t, model_name=f'{best_model_name} (optimal threshold)')

## 9. Feature Importance (XGBoost)

In [ ]:
from src.preprocessing import ALL_FEATURE_COLS

feature_names = ALL_FEATURE_COLS
importances = xgb.named_steps['clf'].feature_importances_

feat_imp = (pd.DataFrame({'feature': feature_names, 'importance': importances})
              .sort_values('importance', ascending=False)
              .head(15))

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=feat_imp, x='importance', y='feature', ax=ax, palette='viridis')
ax.set_title('XGBoost — Top 15 Feature Importances')
ax.set_xlabel('Importance Score')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../data/processed/fig_feature_importance.png', dpi=150)
plt.show()

## 10. Generate Kaggle Submission

Refit the best model on the full training set (no val holdout) and predict probabilities on the test set.

In [ ]:
# Map model name back to pipeline
name_to_pipe = {
    'LR Baseline':       lr_baseline,
    'LR Cost-Sensitive': lr_balanced,
    'XGBoost':           xgb,
}

best_pipe = name_to_pipe[best_model_name]

# Final fit on all training data
best_pipe.fit(X, y)

test_proba = best_pipe.predict_proba(X_test)[:, 1]
make_submission(test_ids, test_proba, SUB_PATH)

pd.read_csv(SUB_PATH).head()

## 11. Save Best Model

In [ ]:
import os
os.makedirs(MODEL_DIR, exist_ok=True)

model_filename = f"{MODEL_DIR}{best_model_name.lower().replace(' ', '_')}_pipeline.pkl"
joblib.dump(best_pipe, model_filename)
print(f'Model saved → {model_filename}')

## Summary

| Model | CV ROC-AUC | Strengths |
|---|---|---|
| LR Baseline | see above | Interpretable coefficients, fast |
| LR Cost-Sensitive | see above | Better recall on churners |
| XGBoost | see above | Captures non-linear feature interactions |

**Key modeling decisions**:
- Stratified K-Fold preserves class ratios — more reliable AUC estimates than random splits
- `class_weight='balanced'` / `scale_pos_weight` aligns training loss with business cost structure
- Threshold optimisation (Section 8) shifts the operating point from ROC-AUC maximisation to cost minimisation — a meaningful distinction for deployment
- Submission uses probability scores (not hard labels) as required by the Kaggle competition